# Apollo Justice v1

Matched-organism contextual self-consistency detector. Each complete conversation is returned to the exact base model and LoRA organism that generated it. The notebook appends the frozen independent-auditor question, preserves the source reasoning trace whenever supplied, and performs no context truncation.

The continuous deception score is `softmax(logit_0, logit_1)[1]` immediately after `Prediction:`. `1` means at least one claim was false or materially misleading. Requests use one optimized NDIF session, length-aware `48/32/16` batches, `logits_to_keep=1`, and literal single-token `0|1` labels.

**Competition category: white-box.** The method reads output logits from the generating organism and should be submitted with `--tag white`.

In [ ]:
import os
import sys

import numpy as np

sys.path.insert(0, "submission")
from apollo_justice import (
    has_source_reasoning,
    query_auditor_scores,
    render_auditor_prompt,
    validate_exact_organism,
)
import util

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = (
    int(os.environ["ALETHEIA_LIMIT"])
    if os.environ.get("ALETHEIA_LIMIT")
    else None
)
SUBMISSION_THRESHOLD = float(os.environ.get("APOLLO_THRESHOLD", "0.5"))

print(
    f"method=apollo_justice_v1 dataset={DATASET_NAME} "
    f"limit={LIMIT} threshold={SUBMISSION_THRESHOLD} category=white"
)

In [ ]:
examples = util.load_examples(DATASET_NAME)
n = len(examples) if LIMIT is None else min(LIMIT, len(examples))
if n < 1:
    raise ValueError("selected dataset is empty")

rows = [examples[position] for position in range(n)]
indices = [row["index"] for row in rows]
model_id, lora_id = validate_exact_organism(rows)
model = util.build_model(model_id, lora_id)
tokenizer = model.tokenizer
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = [row["messages"] for row in rows]
prompts = [render_auditor_prompt(tokenizer, value) for value in messages]
reasoning_rows = sum(has_source_reasoning(value) for value in messages)
print(
    f"model={model_id} lora={lora_id or 'base'} rows={n} "
    f"source_reasoning_rows={reasoning_rows} truncation=disabled"
)

In [ ]:
scores, prompt_lengths, position_batches = query_auditor_scores(
    model,
    tokenizer,
    prompts,
)
if len(scores) != n or not np.isfinite(scores).all():
    raise RuntimeError("Apollo Justice returned missing or non-finite scores")

deceptive = scores >= SUBMISSION_THRESHOLD
print(
    f"prompt_tokens_min={min(prompt_lengths)} "
    f"prompt_tokens_max={max(prompt_lengths)} batches={len(position_batches)} "
    f"score_min={scores.min():.6f} score_mean={scores.mean():.6f} "
    f"score_max={scores.max():.6f} unique_scores={len(np.unique(scores))} "
    f"binary_positives={int(deceptive.sum())}"
)
util.write_submission(
    indices,
    scores,
    threshold=SUBMISSION_THRESHOLD,
    deceptive=deceptive,
)